# Forced alignment — when each line was said

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/espnet/notebook/blob/master/Demos/s2t_align_demo.ipynb) [![s2t_align_demo](https://github.com/espnet/notebook/actions/workflows/s2t_align_demo.yml/badge.svg)](https://github.com/espnet/notebook/actions/workflows/s2t_align_demo.yml)

You have a recording and the text of what was said. Alignment says *when*:
a start and an end for every line, a time for every token inside it, and a
score for how well the line and the audio agree. That is what makes
subtitles, what cuts a long recording into training utterances, and what
throws away the pairs where the text does not match the audio.

Nothing is trained here: alignment reads the CTC head of a model that
already exists. CPU is enough. The checkpoint is 4 GB and cached after the
first run.

## Install

In [ ]:
%pip install -q "espnet==202610.post2" espnet_model_zoo librosa matplotlib

## A recording, and what was said

Four utterances, in the order they were spoken. Alignment needs the order;
it does not need the times, which is the whole point.

In [ ]:
import librosa
from IPython.display import Audio, display

!wget -q -O sample.wav https://github.com/espnet/espnet/raw/master/test_utils/ctc_align_test.wav
speech, rate = librosa.load("sample.wav", sr=16000)
display(Audio(speech, rate=rate))

utterances = [
    "The sale of the hotels",
    "is part of Holiday's strategy",
    "to sell off assets",
    "and concentrate on property management",
]
print(f"{len(speech) / rate:.1f} s of audio, {len(utterances)} utterances")

## Aligning

`ForcedAligner` takes any published checkpoint with a CTC head — an OWSM
one here, but an ASR model works the same way — and returns one segment an
utterance: when it starts, when it ends, and how sure the model is.

In [ ]:
from espnet2.bin.align import ForcedAligner

aligner = ForcedAligner.from_pretrained("espnet/owsm_ctc_v4_1B", device="cpu")
segments = aligner("sample.wav", utterances)

for s in segments:
    print(f"{s.start:5.2f} {s.end:5.2f}  {s.score:.3f}  {s.text}")

## Inside an utterance

Each segment carries the tokens it was made of, with a time and a
probability each. Word-level timestamps come from here.

In [ ]:
for token in segments[0].tokens:
    print(f"{token.start:5.2f} {token.end:5.2f}  {token.score:.3f}  {token.text}")

## Seeing it

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, ax = plt.subplots(figsize=(11, 3))
ax.plot(np.arange(len(speech)) / rate, speech, linewidth=0.4, color="#888")
for i, s in enumerate(segments):
    ax.axvspan(s.start, s.end, color=f"C{i}", alpha=0.25)
    ax.text(
        (s.start + s.end) / 2,
        0.85 * speech.max(),
        s.text.split()[0],
        ha="center",
        fontsize=8,
    )
ax.set_xlabel("seconds")
ax.set_yticks([])
plt.show()

## The score is the point

`score` is the mean probability of an utterance's tokens: 1.0 is a perfect
match. Text that does not belong to the audio still gets aligned — the
algorithm has to put it somewhere — and the score is how you find out.

This is alignment-score filtering, which is how OWSM v4 cleaned its training
data: align, then drop the pairs that score badly
([recipe](https://github.com/espnet/espnet/tree/master/egs2/owsm_v4/s2t1)).

In [ ]:
wrong = list(utterances)
wrong[1] = "the weather today is cold"

for s in aligner("sample.wav", wrong):
    print(f"{s.score:.3f}  {s.text}")

## Write the text the way the model writes it

The score is a probability under *this* model, so the text has to be spelled
the way the model spells it. The same four sentences in capitals — which is
how this recording's reference transcript is written — score zero, because
OWSM's vocabulary has no capitalised words and each one breaks into letters.
The times are still roughly right; the score is not.

In [ ]:
shouted = [u.upper() for u in utterances]
loud = aligner("sample.wav", shouted)

for s in loud:
    print(f"{s.score:.3f}  {s.text}")

# the first line, as the model reads it
print([t.text for t in loud[0].tokens[:8]])

## Where next

- **From the terminal**: `espnet align sample.wav --text "The sale of the hotels"`
- **From an assistant**: the MCP server offers the same thing as an `align` tool
- **Transcription** with the same checkpoint: [`asr_demo.ipynb`](asr_demo.ipynb)
- **The other algorithm**: `espnet2.bin.ctc_segment` is the
  [`ctc_segmentation`](https://arxiv.org/abs/2007.09127) package's, which
  partitions the timeline rather than marking where each token is, and scores
  in log space. `espnet2/bin/asr_align.py` and `s2t_align.py` are its scripts.